# Bill-of-lading classifier: dataset + baseline Transformer
Part 1 of the architecture-variants experiment. Contents: environment info, synthetic
dataset, tokenizer, baseline model, training with seeds 0/1/2, test evaluation.
Every number printed below comes from running this file.

In [1]:
import json, math, os, platform, random, re, sys, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 0. Environment verification (GPU required)
This notebook stops if CUDA is not available. In Colab: Runtime > Change runtime type > GPU.

In [2]:
# Must be set before the first CUDA call. Used with warn_only determinism below.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

print("Python version         :", sys.version.split()[0])
print("PyTorch version        :", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. This experiment must run on a GPU (Colab: Runtime > Change runtime type > "
        "T4 GPU), then restart and run all cells. It will not fall back to CPU.")

DEVICE = torch.device("cuda")
_p = torch.cuda.get_device_properties(0)
print("GPU name               :", torch.cuda.get_device_name(0))
print("CUDA version (torch)   :", torch.version.cuda)
print("GPU compute capability :", f"{_p.major}.{_p.minor}")
print("GPU memory (GB)        :", round(_p.total_memory / 1e9, 2))

ENV = {"python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
       "device": str(DEVICE), "cuda_available": True, "gpu_name": torch.cuda.get_device_name(0),
       "cuda_version": torch.version.cuda, "compute_capability": f"{_p.major}.{_p.minor}",
       "gpu_memory_gb": round(_p.total_memory / 1e9, 2)}

# Best-effort determinism. GPU kernels are not guaranteed bitwise identical to the CPU run
# (or across GPU types), so expect small numeric differences from the CPU reference results.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

Python version         : 3.13.15
PyTorch version        : 2.11.0+cu130
torch.cuda.is_available(): True
GPU name               : Tesla T4
CUDA version (torch)   : 13.0
GPU compute capability : 7.5
GPU memory (GB)        : 15.64


## 1. Synthetic bill-of-lading dataset
Six classes. Two are rare (hazmat, oversize/heavy), 4% each.

**Input corruption (train, val AND test):** OCR character errors, missing fields,
odd spacing, truncated values, and a conflicting instruction borrowed from another class.
The class label is the underlying shipment type and is never changed by the corruption.

**Label noise:** 3% of *training* labels are flipped to a different random class.
Validation and test labels are clean.

In [3]:
CLASSES = ["general_dry", "retail_pallet", "refrigerated", "fragile_high_value", "hazmat", "oversize_heavy"]
PROPS = [0.34, 0.26, 0.20, 0.12, 0.04, 0.04]
RARE_IDS = [CLASSES.index("hazmat"), CLASSES.index("oversize_heavy")]
N_TOTAL = 15_000
SPLIT = {"train": 0.70, "val": 0.10, "test": 0.20}
DATA_SEED = 42
TRAIN_LABEL_NOISE = 0.03
MAX_LEN = 128

# Corruption settings, fixed before looking at any model results
P_DROP_FIELD = 0.25       # each class-specific field is missing with this probability
P_DROP_COMMODITY = 0.05
P_CONFLICT = 0.15         # an instruction typical of another class is appended
P_TRUNCATE = 0.06         # a field is cut off mid-value
OCR_LEVELS = [0.0, 0.01, 0.03, 0.06]     # per-character error rate, one level per document
SPACE_LEVELS = [0.0, 0.03, 0.10]

NAMES = ["ACME", "NORTHSTAR", "BLUE RIDGE", "PACIFIC", "MIDWEST", "SUMMIT", "HARBOR", "KEYSTONE", "LONE STAR", "ATLAS"]
SUFFIX = ["SUPPLY CO", "DISTRIBUTION", "INDUSTRIES", "FOODS", "LOGISTICS", "MFG", "TRADING", "WHSE"]
CITIES = ["DALLAS TX", "MEMPHIS TN", "RENO NV", "ATLANTA GA", "COLUMBUS OH", "TAMPA FL", "DENVER CO", "NEWARK NJ"]
GENERIC_INSTR = ["CALL BEFORE DELIVERY", "LIFTGATE REQUIRED", "APPT REQUIRED", "DELIVER 8AM-4PM", "DO NOT STACK", "INSIDE DELIVERY"]

CLASS_INSTR = {   # used as the conflicting instruction pool
    0: ["NO SPECIAL HANDLING", "STACKABLE", "DRY VAN"],
    1: ["STORE REPLENISHMENT", "ROUTING GUIDE COMPLIANT", "ASN SENT"],
    2: ["KEEP FROZEN", "DO NOT FREEZE", "REEFER CONTINUOUS"],
    3: ["FRAGILE", "SIGNATURE REQUIRED", "HANDLE WITH CARE", "THIS SIDE UP"],
    4: ["HAZMAT", "PLACARD REQUIRED", "FLAMMABLE"],
    5: ["OVERSIZE LOAD", "ESCORT REQUIRED", "FLATBED TARP"],
}


def _party(r):
    return f"{r.choice(NAMES)} {r.choice(SUFFIX)} {r.choice(CITIES)}"


def make_fields(c, r):
    """Returns (always_present_fields, droppable_class_fields, commodity_field)."""
    base = [f"BOL {r.randint(100000, 999999)}", f"DATE {r.randint(1, 12):02d}/{r.randint(1, 28):02d}/25",
            f"SHIPPER {_party(r)}", f"CONSIGNEE {_party(r)}", f"CARRIER {r.choice(NAMES)} FREIGHT",
            r.choice(["PREPAID", "COLLECT", "3RD PARTY"])]
    sig, com = [], None
    pcs = r.randint(1, 40)
    if c == 0:
        com = "DESC " + r.choice(["PAPER PRODUCTS", "STEEL BRACKETS", "EMPTY PLASTIC BOTTLES", "CARDBOARD CARTONS",
                                  "PVC FITTINGS", "FASTENERS", "CERAMIC TILE", "LUMBER", "TIRES"])
        sig = [f"{pcs} {r.choice(['PLT', 'CTN', 'BDL'])}", f"WT {r.randint(2000, 44000)} LB",
               f"FREIGHT CLASS {r.choice([50, 55, 60, 65, 70, 77.5, 85])}", f"NMFC {r.randint(10000, 160000)}",
               r.choice(CLASS_INSTR[0])]
    elif c == 1:
        com = "DESC " + r.choice(["APPAREL", "HOUSEWARES", "SMALL APPLIANCES", "TOYS", "HOME TEXTILES", "PET SUPPLIES"])
        sig = [f"{pcs} PLT", f"WT {r.randint(2000, 30000)} LB", f"PO {r.randint(10000000, 99999999)}",
               f"DC {r.randint(1, 99):02d}", r.choice(["SSCC LABELS APPLIED", "PALLET LABELS ATTACHED"]),
               r.choice(CLASS_INSTR[1]), "DELIVERY APPT REQUIRED"]
    elif c == 2:
        frozen = r.random() < 0.5
        com = "DESC " + r.choice(["FROZEN POULTRY", "ICE CREAM", "FROZEN SEAFOOD"] if frozen
                                 else ["FRESH PRODUCE", "DAIRY", "CHILLED MEAT", "FLORAL"])
        sig = [f"{pcs} PLT", f"WT {r.randint(8000, 42000)} LB",
               f"SET TEMP {r.choice([-20, -18, -15]) if frozen else r.choice([34, 36, 38])} {'C' if frozen else 'F'}",
               "REEFER", r.choice(CLASS_INSTR[2])]
    elif c == 3:
        com = "DESC " + r.choice(["LCD MONITORS", "GLASSWARE", "LAPTOPS", "FRAMED ARTWORK", "MEDICAL DEVICES", "JEWELRY"])
        sig = [f"{pcs} CTN", f"WT {r.randint(200, 9000)} LB", f"DECLARED VALUE ${r.randint(20, 900) * 1000}",
               r.choice(CLASS_INSTR[3]), r.choice(CLASS_INSTR[3])]
    elif c == 4:
        com = "DESC " + r.choice(["GASOLINE", "FLAMMABLE LIQUID NOS", "PAINT", "LITHIUM ION BATTERIES", "ETHANOL SOLUTION",
                                  "NITRIC ACID", "SODIUM HYDROXIDE SOLUTION", "AEROSOLS"])
        sig = [f"{pcs} DRM", f"WT {r.randint(500, 30000)} LB", f"UN{r.choice([1203, 1993, 1263, 3480, 1170, 2031, 1824, 1950])}",
               f"CLASS {r.choice(['3', '8', '9', '2.1', '4.1'])}", f"PG {r.choice(['II', 'III'])}",
               "PLACARD REQUIRED", f"24HR EMERGENCY CONTACT {r.randint(200, 999)}-{r.randint(200, 999)}-{r.randint(1000, 9999)}"]
    else:
        com = "DESC " + r.choice(["EXCAVATOR", "CNC MACHINE", "TRANSFORMER", "WIND BLADE SECTION", "INDUSTRIAL BOILER",
                                  "STEEL BEAMS", "PRECAST CONCRETE"])
        sig = ["1 PC", f"WT {r.randint(42000, 140000)} LB", f"DIMS {r.randint(240, 600)}X{r.randint(96, 160)}X{r.randint(96, 170)} IN",
               "OVERSIZE LOAD", f"PERMIT {r.randint(100000, 999999)}", "ESCORT REQUIRED", "FLATBED TARP"]
    return base, sig, com


OCR_MAP = {"O": "0", "0": "O", "I": "1", "1": "I", "S": "5", "5": "S", "B": "8", "8": "B", "E": "F", "M": "N", "L": "I"}


def ocr_corrupt(s, p, r):
    out = []
    for ch in s:
        u = r.random()
        if ch in OCR_MAP and u < p:
            out.append(OCR_MAP[ch])
        elif ch.isalnum() and u < 1.5 * p:      # dropped character
            continue
        else:
            out.append(ch)
    return "".join(out)


def space_corrupt(s, p, r):
    out = []
    for ch in s:
        if ch == " " and r.random() < p:
            out.append(r.choice(["  ", "   ", ""]))
        elif ch.isalpha() and r.random() < p / 4:
            out.append(ch + " ")
        else:
            out.append(ch)
    return "".join(out)


def make_doc(c, r):
    base, sig, com = make_fields(c, r)
    sig = [f for f in sig if r.random() > P_DROP_FIELD]
    if com is not None and r.random() > P_DROP_COMMODITY:
        sig.insert(0, com)
    extra = [r.choice(GENERIC_INSTR)] if r.random() < 0.5 else []
    if r.random() < P_CONFLICT:
        other = r.choice([k for k in range(len(CLASSES)) if k != c])
        extra.append(r.choice(CLASS_INSTR[other]))
    fields = base + sig + extra
    # keep header fields first, shuffle the rest lightly so order is not a perfect cue
    head, tail = fields[:6], fields[6:]
    r.shuffle(tail)
    fields = head + tail
    fields = [f[: max(3, int(len(f) * r.uniform(0.3, 0.9)))] if r.random() < P_TRUNCATE else f for f in fields]
    text = r.choice([" ", " | ", " ; ", "\n"]).join(fields)
    text = ocr_corrupt(text, r.choice(OCR_LEVELS), r)
    return space_corrupt(text, r.choice(SPACE_LEVELS), r)


def build_dataset(seed=DATA_SEED):
    r = random.Random(seed)
    counts = [round(p * N_TOTAL) for p in PROPS]
    splits = {k: ([], []) for k in SPLIT}
    for c, n in enumerate(counts):
        n_tr, n_va = round(n * SPLIT["train"]), round(n * SPLIT["val"])
        bounds = {"train": (0, n_tr), "val": (n_tr, n_tr + n_va), "test": (n_tr + n_va, n)}
        for k, (a, b) in bounds.items():
            for _ in range(b - a):
                splits[k][0].append(make_doc(c, r))
                splits[k][1].append(c)
    data = {}
    for k, (texts, labels) in splits.items():
        idx = list(range(len(texts)))
        r.shuffle(idx)
        data[k] = ([texts[i] for i in idx], np.array([labels[i] for i in idx]))
    # label noise on TRAIN labels only
    tr_texts, y_clean = data["train"]
    y_noisy = y_clean.copy()
    nr = np.random.RandomState(seed)
    flip = nr.rand(len(y_noisy)) < TRAIN_LABEL_NOISE
    for i in np.where(flip)[0]:
        y_noisy[i] = nr.choice([k for k in range(len(CLASSES)) if k != y_clean[i]])
    data["train"] = (tr_texts, y_noisy)
    return data, y_clean, flip


data, y_train_clean, flipped = build_dataset()

# Fingerprint of the exact dataset (texts + labels, all splits). If this differs from the reference
# below, the dataset on this machine is NOT the one used in the earlier CPU run.
import hashlib
_h = hashlib.sha256()
for _k in ("train", "val", "test"):
    _h.update(json.dumps([data[_k][0], data[_k][1].tolist()]).encode())
DATA_SHA256 = _h.hexdigest()
REFERENCE_DATA_SHA256 = "e74c4242ce78d13fec8ad56ebe0647c6598766c815068a00ccb2369c6307d610"  # from the earlier CPU run
REFERENCE_VOCAB = 5565
print("dataset sha256:", DATA_SHA256)
if DATA_SHA256 != REFERENCE_DATA_SHA256:
    raise RuntimeError("Dataset differs from the reference (data seed 42) dataset. Stopping so a different "
                       "experiment is not run by accident.")
print("dataset matches the reference dataset: OK")
print("Example documents:")
for c in range(len(CLASSES)):
    i = int(np.where(data["test"][1] == c)[0][0])
    print(f"[{CLASSES[c]}]", data["test"][0][i].replace("\n", " / ")[:230])
print()
dist = {k: np.bincount(v[1], minlength=6).tolist() for k, v in data.items()}
print("class counts per split", dist)
print(f"train label noise: {int(flipped.sum())} of {len(flipped)} labels flipped ({flipped.mean():.2%}); val/test labels are clean")

dataset sha256: e74c4242ce78d13fec8ad56ebe0647c6598766c815068a00ccb2369c6307d610
dataset matches the reference dataset: OK
Example documents:
[general_dry] BOL 9597 / DATE 07/04/25 / SHIPP ER MIDWEST WHSE ATLANTA GA / CONSIGNEE KEYSTONE TADING MEPHIS T / CARRIER BLUE RIDGE FREIGHT / 3RD PARTY / INSIDE DE LIVERY / NMFC 54732 / FREIGHT CLSS 65 / DESC LMBER  / STACKABLE / WT 19471 LB
[retail_pallet] BOL  722203 DATE 08/06/25 SHIPP ER MIDWEST WHSE RENO NV CONSIGNEE BLUE RIDGE INDUSTRIE S COLUMBUS OH CARRIER BLUE RIDGE FREIGHT COLLECT DESC HOME TEXILES PALLET LABELS ATTACHED  LIFTGATE REQUIRED DELIVERY APPT REQUIRED WT 11559 LB
[refrigerated] BOL 364476; ATE 03/01/25 ; SHIPER KEYSTONE ID U STRIES  RENO NV; CONSIGNE E HARBOR FOODSCOLUMBUS OH ; CARRIER MIDWESTFEIHT ; COLLECT   ; 15PLT ; WT 2011 LB ; REEFER ; DESC ICE CREAM ; EEFER  CONTIUO U ;SET TEMP  -20 C
[fragile_high_value] BOL 819343 | DATE 04/21/25 | SHIPPER MIDWEST FOODS MEMPHIS TN | CONSIGNEE   PACIFIC WHSE RENO NV | CARRIER HARBORFR

## 2. Tokenizer
Lowercase, split into alphanumeric runs and single punctuation marks. Vocabulary is built from the
training texts only (min frequency 2). Fixed length 128, pad id 0, unknown id 1.

In [4]:
TOKEN_RE = re.compile(r"[a-z0-9]+|[^\sa-z0-9]")
PAD, UNK = 0, 1


def tok(s):
    return TOKEN_RE.findall(s.lower())


counter = Counter(t for s in data["train"][0] for t in tok(s))
itos = ["<pad>", "<unk>"] + [t for t, n in counter.most_common() if n >= 2]
stoi = {t: i for i, t in enumerate(itos)}
VOCAB = len(itos)


def encode(texts):
    ids = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    lens, trunc = [], 0
    for i, s in enumerate(texts):
        t = [stoi.get(w, UNK) for w in tok(s)]
        trunc += len(t) > MAX_LEN
        t = t[:MAX_LEN]
        ids[i, : len(t)] = t
        lens.append(len(t))
    return torch.from_numpy(ids), np.array(lens), trunc


X, Y, TOKSTATS = {}, {}, {}
for k, (texts, labels) in data.items():
    ids, lens, trunc = encode(texts)
    X[k], Y[k] = ids, torch.from_numpy(labels)
    TOKSTATS[k] = {"n_docs": len(texts), "mean_len": float(lens.mean()), "max_len_before_trunc_cap": int(lens.max()),
                   "pct_truncated": 100 * trunc / len(texts), "nonpad_tokens": int(lens.sum())}
unk_rate = {k: float((X[k] == UNK).sum() / (X[k] != PAD).sum()) for k in X}
print("vocab size:", VOCAB)
assert VOCAB == REFERENCE_VOCAB, f"vocab size {VOCAB} != reference {REFERENCE_VOCAB}"
print("token stats:", json.dumps(TOKSTATS, indent=1))
print("unk rate:", {k: round(v, 4) for k, v in unk_rate.items()})

vocab size: 5565
token stats: {
 "train": {
  "n_docs": 10500,
  "mean_len": 43.58723809523809,
  "max_len_before_trunc_cap": 70,
  "pct_truncated": 0.0,
  "nonpad_tokens": 457666
 },
 "val": {
  "n_docs": 1500,
  "mean_len": 43.326,
  "max_len_before_trunc_cap": 69,
  "pct_truncated": 0.0,
  "nonpad_tokens": 64989
 },
 "test": {
  "n_docs": 3000,
  "mean_len": 43.673,
  "max_len_before_trunc_cap": 72,
  "pct_truncated": 0.0,
  "nonpad_tokens": 131019
 }
}
unk rate: {'train': 0.0552, 'val': 0.0637, 'test': 0.064}


## 3. Baseline model
Embedding + learned positions, 4 pre-LN Transformer blocks (d=128, 4 heads, FFN 256),
padding-aware softmax attention, mean pooling over non-padding tokens, linear head.
The attention is its own module so a later variant can swap it out.

In [5]:
D_MODEL, N_LAYERS, N_HEADS, D_FF = 128, 4, 4, 256


class SoftmaxSelfAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)

    def forward(self, x, pad_mask):          # pad_mask: (B, T) True where real token
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        key_mask = pad_mask[:, None, None, :]                       # padded keys are never attended to
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        return self.out(o.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, attn):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.attn = attn
        self.ff = nn.Sequential(nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL))

    def forward(self, x, pad_mask):
        x = x + self.attn(self.ln1(x), pad_mask)
        return x + self.ff(self.ln2(x))


class Classifier(nn.Module):
    def __init__(self, vocab, n_classes, attn_factory):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, D_MODEL, padding_idx=PAD)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)
        self.blocks = nn.ModuleList([Block(attn_factory()) for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, n_classes)

    def forward(self, ids):
        pad_mask = ids != PAD
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(ids.shape[1], device=ids.device))
        for b in self.blocks:
            x = b(x, pad_mask)
        x = self.ln_f(x)
        m = pad_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1)               # mean over real tokens only
        return self.head(pooled)


def build_baseline():
    return Classifier(VOCAB, len(CLASSES), lambda: SoftmaxSelfAttention(D_MODEL, N_HEADS))


def param_counts(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.tok_emb.weight.numel() + model.pos_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# Sanity check: padding must not change the output. Logits with the full 128 positions should match
# logits when the trailing all-padding columns are cut off (so padding is masked in attention and pooling).
_m = build_baseline().eval()
_ids = X["val"][:8]
_L = int((_ids != PAD).sum(1).max())
with torch.no_grad():
    _diff = (_m(_ids) - _m(_ids[:, :_L])).abs().max().item()
print(f"padding invariance: max |logit diff| = {_diff:.2e} (cut to {_L} of {MAX_LEN} positions)")
assert _diff < 1e-4, "padding is leaking into the output"
print("param counts:", param_counts(_m))

padding invariance: max |logit diff| = 5.96e-08 (cut to 57 of 128 positions)
param counts: {'total': 1259654, 'embedding': 728704, 'non_embedding': 530950}


## 4. Training
AdamW, lr 1e-3, weight decay 0.01, 100 warmup steps then linear decay to 0, batch 64, 8 epochs,
plain cross-entropy (no class weights, no resampling), fp32. The model after the last epoch is the model
that gets evaluated. Validation is only monitored. Test is evaluated once, after training.

In [6]:
EPOCHS, BATCH, LR, WD, WARMUP = 8, 64, 1e-3, 0.01, 100
SEEDS = [0, 1, 2]


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, ids, bs=256):
    model.eval()
    return torch.cat([model(ids[i:i + bs].to(DEVICE)).argmax(-1).cpu() for i in range(0, len(ids), bs)]).numpy()


def metrics(y_true, y_pred):
    per_class = {}
    for c, name in enumerate(CLASSES):
        m = y_true == c
        per_class[name] = float((y_pred[m] == c).mean())      # per-class accuracy = recall of that class
    rare = [per_class[CLASSES[i]] for i in RARE_IDS]
    return {"overall_acc": float((y_true == y_pred).mean()), "per_class_acc": per_class,
            "rare_class_recall_macro": float(np.mean(rare)),
            "rare_support": {CLASSES[i]: int((y_true == i).sum()) for i in RARE_IDS}}


def train_baseline(seed):
    set_seed(seed)
    model = build_baseline().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n = len(X["train"])
    steps_per_epoch = math.ceil(n / BATCH)
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / WARMUP if s < WARMUP else max(0.0, (total - s) / (total - WARMUP)))
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = X["train"], Y["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    log, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        run, cnt = 0.0, 0
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = xtr[idx].to(DEVICE), ytr[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            run += loss.item() * len(idx); cnt += len(idx)
        val_acc = float((predict(model, X["val"]) == Y["val"].numpy()).mean())
        log.append({"epoch": ep + 1, "train_loss": run / cnt, "val_acc": val_acc})
        print(f"  seed {seed} epoch {ep + 1}/{EPOCHS} train_loss {run / cnt:.4f} val_acc {val_acc:.4f}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return model, log, time.time() - t0

In [7]:
results = {"env": ENV, "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "wd": WD, "warmup": WARMUP,
                                  "max_len": MAX_LEN, "d_model": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS,
                                  "d_ff": D_FF, "vocab": VOCAB, "train_label_noise": TRAIN_LABEL_NOISE,
                                  "data_seed": DATA_SEED, "data_sha256": DATA_SHA256},
           "class_counts": dist, "token_stats": TOKSTATS, "baseline": {}}
y_test = Y["test"].numpy()
for seed in SEEDS:
    model, log, secs = train_baseline(seed)
    pred = predict(model, X["test"])
    r = metrics(y_test, pred)
    r.update(train_seconds=secs, epoch_log=log, params=param_counts(model),
             val_acc_final=log[-1]["val_acc"])
    results["baseline"][f"seed{seed}"] = r
    print(f"seed {seed}: test acc {r['overall_acc']:.4f} | rare macro recall {r['rare_class_recall_macro']:.4f} | {secs:.0f}s")
    torch.save(model.state_dict(), f"{OUT_DIR}/baseline_seed{seed}.pt")

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  seed 0 epoch 1/8 train_loss 0.8361 val_acc 0.9720
  seed 0 epoch 2/8 train_loss 0.2317 val_acc 0.9840
  seed 0 epoch 3/8 train_loss 0.1903 val_acc 0.9767
  seed 0 epoch 4/8 train_loss 0.1724 val_acc 0.9847
  seed 0 epoch 5/8 train_loss 0.1517 val_acc 0.9867
  seed 0 epoch 6/8 train_loss 0.1334 val_acc 0.9833
  seed 0 epoch 7/8 train_loss 0.1124 val_acc 0.9747
  seed 0 epoch 8/8 train_loss 0.0965 val_acc 0.9767
seed 0: test acc 0.9750 | rare macro recall 0.9458 | 36s
  seed 1 epoch 1/8 train_loss 0.8163 val_acc 0.9760
  seed 1 epoch 2/8 train_loss 0.2361 val_acc 0.9827
  seed 1 epoch 3/8 train_loss 0.1924 val_acc 0.9740
  seed 1 epoch 4/8 train_loss 0.1697 val_acc 0.9827
  seed 1 epoch 5/8 train_loss 0.1512 val_acc 0.9847
  seed 1 epoch 6/8 train_loss 0.1320 val_acc 0.9793
  seed 1 epoch 7/8 train_loss 0.1184 val_acc 0.9827
  seed 1 epoch 8/8 train_loss 0.0966 val_acc 0.9800
seed 1: test acc 0.9783 | rare macro recall 0.9667 | 33s
  seed 2 epoch 1/8 train_loss 0.8317 val_acc 0.9727
  

## 5. Baseline report (seeds 0, 1, 2)

In [8]:
import pandas as pd

HAZ, OVR = "hazmat", "oversize_heavy"
runs = results["baseline"]
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"])

per_seed = pd.DataFrame({
    sd: {"train_seconds": r["train_seconds"], "final_val_acc": r["val_acc_final"],
         "overall_test_acc": r["overall_acc"],
         **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
         "rare_macro_recall": r["rare_class_recall_macro"]}
    for sd, r in runs.items()})
print(per_seed.round(4).to_string())

p = runs["seed0"]["params"]
print(f"\nparams: total {p['total']:,} | embedding {p['embedding']:,} | non-embedding {p['non_embedding']:,}")

agg = pd.DataFrame({"overall_test_acc": per_seed.loc["overall_test_acc"],
                    "hazmat_recall": per_seed.loc[f"acc_{HAZ}"],
                    "oversize_recall": per_seed.loc[f"acc_{OVR}"],
                    "rare_macro_recall": per_seed.loc["rare_macro_recall"]}).T
agg["mean"], agg["std"] = agg.mean(axis=1), agg.std(axis=1, ddof=1)
print("\nmean and sample std (ddof=1) over 3 seeds")
print(agg[["mean", "std"]].round(4).to_string())

results["summary"] = {"mean": agg["mean"].to_dict(), "std": agg["std"].to_dict()}
with open(f"{OUT_DIR}/baseline_results_cuda.json", "w") as f:
    json.dump(results, f, indent=1)

GPU: Tesla T4 | CUDA 13.0 | cc 7.5
                          seed0    seed1    seed2
train_seconds           35.7705  32.9791  34.6759
final_val_acc            0.9767   0.9800   0.9780
overall_test_acc         0.9750   0.9783   0.9757
acc_general_dry          0.9804   0.9824   0.9755
acc_retail_pallet        0.9718   0.9769   0.9705
acc_refrigerated         0.9833   0.9833   0.9750
acc_fragile_high_value   0.9722   0.9694   0.9833
acc_hazmat               0.9333   0.9417   0.9750
acc_oversize_heavy       0.9583   0.9917   0.9917
rare_macro_recall        0.9458   0.9667   0.9833

params: total 1,259,654 | embedding 728,704 | non-embedding 530,950

mean and sample std (ddof=1) over 3 seeds
                     mean     std
overall_test_acc   0.9763  0.0018
hazmat_recall      0.9500  0.0220
oversize_recall    0.9806  0.0192
rare_macro_recall  0.9653  0.0188


# Part 2: Linear-attention variant
Only the sequence-mixing layer changes. `Classifier`, `Block`, the dataset, tokenizer, optimizer,
schedule and seeds are the same objects the baseline used. The baseline cells above are not re-run here.

**How to run:** run this section in the same Colab session after the baseline cells. If the runtime
was restarted, run the setup cells (0 to the model definitions) but skip the baseline training cell;
the comparison cell then reads `results/baseline_results_cuda.json`.

**Methodology limitation (kept from Part 1):** CUDA kernels are not guaranteed to be bitwise
deterministic. Determinism is requested with `warn_only=True`. In the baseline run on the T4, PyTorch warned that
"Memory Efficient attention defaults to a non-deterministic algorithm" (softmax attention's backward pass).
The baseline was not changed or rerun because of this. The linear-attention block uses `einsum` and
elementwise ops instead of that kernel, so the warning may not appear for it. The two models therefore
may not have the same degree of run-to-run GPU nondeterminism, and a different GPU can shift numbers slightly.
Fixed seeds, the identical dataset, and identical initial weights per seed reduce this variation but do not remove it.
Latency is not measured in this section, and no speed claim is made for linear attention.


In [9]:
# --- Linear attention: phi(x) = ELU(x) + 1, non-causal, padding-aware -----------------------
class LinearSelfAttention(nn.Module):
    """Same parameters as SoftmaxSelfAttention (qkv and out projections), different mixing.

    out_i = phi(q_i)^T (sum_j phi(k_j) v_j^T) / (phi(q_i)^T sum_j phi(k_j) + eps)
    Padded keys/values are zeroed before the sums, so they contribute nothing. No T x T matrix is built:
    the sums are (dh x dh) per head.
    """
    def __init__(self, d, h, eps=1e-6):
        super().__init__()
        self.h, self.dh, self.eps = h, d // h, eps
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)

    def forward(self, x, pad_mask):          # x: (B, T, D), pad_mask: (B, T) True = real token
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)   # (B, h, T, dh)
        q, k = F.elu(q) + 1, F.elu(k) + 1
        m = pad_mask[:, None, :, None].to(k.dtype)                                     # (B, 1, T, 1)
        k, v = k * m, v * m                                                            # padded keys/values -> 0
        kv = torch.einsum("bhtd,bhte->bhde", k, v)                                     # sum_j phi(k_j) v_j^T
        z = k.sum(dim=2)                                                               # sum_j phi(k_j)
        num = torch.einsum("bhtd,bhde->bhte", q, kv)
        den = torch.einsum("bhtd,bhd->bht", q, z).unsqueeze(-1) + self.eps
        o = (num / den).transpose(1, 2).reshape(B, T, D)
        return self.out(o)


def build_linear():
    return Classifier(VOCAB, len(CLASSES), lambda: LinearSelfAttention(D_MODEL, N_HEADS))

In [10]:
# --- Parameter comparison (stop if more than 2% apart) ---------------------------------------
set_seed(0); _base_m = build_baseline()
set_seed(0); _lin_m = build_linear()
pb, pl = param_counts(_base_m), param_counts(_lin_m)
diff = abs(pl["total"] - pb["total"])
pct = 100 * diff / pb["total"]
print(f"Baseline total parameters         : {pb['total']:,}")
print(f"Linear-attention total parameters : {pl['total']:,}")
print(f"Absolute difference               : {diff:,}")
print(f"Percentage difference             : {pct:.4f}%")
if pct > 2.0:
    raise RuntimeError(f"Parameter counts differ by {pct:.2f}% (> 2%). Stopping.")
print("Embedding / non-embedding (linear):", pl["embedding"], "/", pl["non_embedding"])

# Same seed + same parameter shapes/order -> identical initial weights, so seed s starts from the same point
_sb, _sl = _base_m.state_dict(), _lin_m.state_dict()
INIT_IDENTICAL = list(_sb) == list(_sl) and all(torch.equal(_sb[k], _sl[k]) for k in _sb)
print("Initial weights identical to baseline for the same seed:", INIT_IDENTICAL)

Baseline total parameters         : 1,259,654
Linear-attention total parameters : 1,259,654
Absolute difference               : 0
Percentage difference             : 0.0000%
Embedding / non-embedding (linear): 728704 / 530950
Initial weights identical to baseline for the same seed: True


In [11]:
# --- Correctness tests (run on DEVICE, before training) -------------------------------------
def naive_linear_attention(attn, x, pad_mask):
    """Reference only: explicit T x T weights with the same math. Used to check the fast version."""
    B, T, D = x.shape
    q, k, v = attn.qkv(x).view(B, T, 3, attn.h, attn.dh).permute(2, 0, 3, 1, 4)
    q, k = F.elu(q) + 1, F.elu(k) + 1
    A = torch.einsum("bhtd,bhsd->bhts", q, k) * pad_mask[:, None, None, :].to(q.dtype)
    A = A / (A.sum(-1, keepdim=True) + attn.eps)
    return attn.out((A @ v).transpose(1, 2).reshape(B, T, D))


@torch.no_grad()
def padding_invariance(model, ids):
    """Max |logit diff| between the full 128-wide batch and versions with trailing padding removed."""
    model.eval()
    ids = ids.to(DEVICE)
    L = int((ids != PAD).sum(1).max())
    full = model(ids)
    return max((full - model(ids[:, :n])).abs().max().item() for n in (L, min(L + 20, MAX_LEN)))


@torch.no_grad()
def attention_matches_reference(attn, ids):
    attn.eval()
    ids = ids.to(DEVICE)
    mask = ids != PAD
    x = torch.randn(ids.shape[0], ids.shape[1], D_MODEL, device=DEVICE)
    return (attn(x, mask) - naive_linear_attention(attn, x, mask)).abs().max().item()


_ids = X["val"][:64]
_lin_m = _lin_m.to(DEVICE)
PAD_TEST_UNTRAINED = padding_invariance(_lin_m, _ids)
REF_TEST = attention_matches_reference(_lin_m.blocks[0].attn, _ids)
print(f"linear padding invariance (untrained): max |logit diff| = {PAD_TEST_UNTRAINED:.2e}")
print(f"fast linear attention vs explicit T x T reference: max |diff| = {REF_TEST:.2e}")
assert PAD_TEST_UNTRAINED < 1e-4, "padding is leaking into the linear-attention output"
assert REF_TEST < 1e-4, "fast linear attention does not match the reference computation"

# Negative control: the same attention with key/value masking switched off must FAIL the padding test.
# This shows the test can detect padding leakage (an exact 0 above is not a blind test).
class _LeakyLinear(LinearSelfAttention):
    def forward(self, x, pad_mask):
        return super().forward(x, torch.ones_like(pad_mask))


_leaky = Classifier(VOCAB, len(CLASSES), lambda: _LeakyLinear(D_MODEL, N_HEADS)).to(DEVICE)
LEAK_CONTROL = padding_invariance(_leaky, _ids)
print(f"negative control (masking off): max |logit diff| = {LEAK_CONTROL:.2e}  (must be > 1e-4)")
assert LEAK_CONTROL > 1e-4, "padding test is too weak to detect leakage"
del _base_m, _lin_m, _sb, _sl, _leaky

linear padding invariance (untrained): max |logit diff| = 1.04e-07
fast linear attention vs explicit T x T reference: max |diff| = 1.19e-07
negative control (masking off): max |logit diff| = 8.75e-02  (must be > 1e-4)


## Training the linear-attention model (seeds 0, 1, 2)
`train_variant` is `train_baseline` with the model constructor passed in. Everything else is identical:
AdamW, lr 1e-3, weight decay 0.01, 100 warm-up steps then linear decay, batch 64, 8 epochs, fp32,
plain cross-entropy, noisy training labels, final-epoch model, test evaluated once after training.

In [12]:
def train_variant(seed, build_fn):
    set_seed(seed)
    model = build_fn().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n = len(X["train"])
    steps_per_epoch = math.ceil(n / BATCH)
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / WARMUP if s < WARMUP else max(0.0, (total - s) / (total - WARMUP)))
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = X["train"], Y["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    log, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        run, cnt = 0.0, 0
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = xtr[idx].to(DEVICE), ytr[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step(); sched.step()
            run += loss.item() * len(idx); cnt += len(idx)
        val_acc = float((predict(model, X["val"]) == Y["val"].numpy()).mean())
        log.append({"epoch": ep + 1, "train_loss": run / cnt, "val_acc": val_acc})
        print(f"  seed {seed} epoch {ep + 1}/{EPOCHS} train_loss {run / cnt:.4f} val_acc {val_acc:.4f}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return model, log, time.time() - t0

In [13]:
TEST_EVAL_CALLS = 0
lin = {"env": ENV, "architecture": "linear_attention_elu_plus_1",
       "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "wd": WD, "warmup": WARMUP,
                  "max_len": MAX_LEN, "d_model": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS,
                  "d_ff": D_FF, "vocab": VOCAB, "train_label_noise": TRAIN_LABEL_NOISE,
                  "data_seed": DATA_SEED, "data_sha256": DATA_SHA256},
       "param_check": {"baseline_total": pb["total"], "linear_total": pl["total"], "pct_diff": pct,
                       "init_identical_to_baseline": INIT_IDENTICAL},
       "padding_test_untrained_max_logit_diff": PAD_TEST_UNTRAINED, "leak_control_max_logit_diff": LEAK_CONTROL, "reference_test_max_diff": REF_TEST,
       "linear": {}}
y_test = Y["test"].numpy()
for seed in SEEDS:
    model, log, secs = train_variant(seed, build_linear)
    TEST_EVAL_CALLS += 1                              # test is touched once per model, after training
    pred = predict(model, X["test"])
    r = metrics(y_test, pred)
    r.update(train_seconds=secs, epoch_log=log, params=param_counts(model), val_acc_final=log[-1]["val_acc"],
             padding_test_trained_max_logit_diff=padding_invariance(model, X["val"][:64]))
    lin["linear"][f"seed{seed}"] = r
    print(f"seed {seed}: test acc {r['overall_acc']:.4f} | rare macro recall {r['rare_class_recall_macro']:.4f} | "
          f"{secs:.0f}s | trained-model padding diff {r['padding_test_trained_max_logit_diff']:.2e}")
    torch.save(model.state_dict(), f"{OUT_DIR}/linear_seed{seed}.pt")     # kept for later benchmarking
    del model

with open(f"{OUT_DIR}/linear_results_cuda.json", "w") as f:
    json.dump(lin, f, indent=1)
print("saved:", [f"{OUT_DIR}/linear_seed{s}.pt" for s in SEEDS], "and linear_results_cuda.json")
print("Download these from Colab (or copy to Drive) before the runtime is recycled.")

  seed 0 epoch 1/8 train_loss 0.8873 val_acc 0.9613
  seed 0 epoch 2/8 train_loss 0.2556 val_acc 0.9733
  seed 0 epoch 3/8 train_loss 0.2047 val_acc 0.9873
  seed 0 epoch 4/8 train_loss 0.1893 val_acc 0.9873
  seed 0 epoch 5/8 train_loss 0.1758 val_acc 0.9853
  seed 0 epoch 6/8 train_loss 0.1655 val_acc 0.9867
  seed 0 epoch 7/8 train_loss 0.1575 val_acc 0.9807
  seed 0 epoch 8/8 train_loss 0.1496 val_acc 0.9793
seed 0: test acc 0.9817 | rare macro recall 0.9625 | 37s | trained-model padding diff 9.54e-07
  seed 1 epoch 1/8 train_loss 0.8685 val_acc 0.9453
  seed 1 epoch 2/8 train_loss 0.2684 val_acc 0.9780
  seed 1 epoch 3/8 train_loss 0.2079 val_acc 0.9720
  seed 1 epoch 4/8 train_loss 0.1883 val_acc 0.9800
  seed 1 epoch 5/8 train_loss 0.1777 val_acc 0.9847
  seed 1 epoch 6/8 train_loss 0.1654 val_acc 0.9840
  seed 1 epoch 7/8 train_loss 0.1581 val_acc 0.9840
  seed 1 epoch 8/8 train_loss 0.1504 val_acc 0.9827
seed 1: test acc 0.9830 | rare macro recall 0.9625 | 36s | trained-model 

## Linear-attention report, and comparison with the baseline

In [14]:
import pandas as pd

if "results" in globals() and len(results.get("baseline", {})) == len(SEEDS):
    base = results
else:
    with open(f"{OUT_DIR}/baseline_results_cuda.json") as f:
        base = json.load(f)
B, L = base["baseline"], lin["linear"]
HAZ, OVR = "hazmat", "oversize_heavy"
print("GPU:", ENV["gpu_name"], "| CUDA", ENV["cuda_version"], "| cc", ENV["compute_capability"])


def per_seed_table(runs):
    return pd.DataFrame({
        sd: {"train_seconds": r["train_seconds"], "final_val_acc": r["val_acc_final"],
             "overall_test_acc": r["overall_acc"],
             **{f"acc_{c}": r["per_class_acc"][c] for c in CLASSES},
             "rare_macro_recall": r["rare_class_recall_macro"]}
        for sd, r in runs.items()})


tl = per_seed_table(L)
print("\nLinear attention, per seed")
print(tl.round(4).to_string())

ROWS = [("overall_test_acc", "overall_test_acc"), ("hazmat_recall", f"acc_{HAZ}"),
        ("oversize_recall", f"acc_{OVR}"), ("rare_macro_recall", "rare_macro_recall")]
tb = per_seed_table(B)
side = pd.DataFrame({name: {"baseline_mean": tb.loc[key].mean(), "baseline_std": tb.loc[key].std(ddof=1),
                            "linear_mean": tl.loc[key].mean(), "linear_std": tl.loc[key].std(ddof=1)}
                     for name, key in ROWS}).T
side["linear_minus_baseline"] = side["linear_mean"] - side["baseline_mean"]
print("\nBaseline vs linear attention: mean and sample std (ddof=1) over seeds 0, 1, 2")
print(side.round(4).to_string())

pc = pd.DataFrame({"baseline_mean": tb.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1),
                   "linear_mean": tl.loc[[f"acc_{c}" for c in CLASSES]].mean(axis=1)})
pc["linear_minus_baseline"] = pc["linear_mean"] - pc["baseline_mean"]
print("\nPer-class test accuracy, mean over seeds")
print(pc.round(4).to_string())
print("\nTraining seconds per seed  baseline:", [round(B[s]["train_seconds"]) for s in B],
      "| linear:", [round(L[s]["train_seconds"]) for s in L])
print("(training wall-clock on a shared Colab GPU; this is not an inference-latency measurement)")
print("Seed-to-seed std is on 3 runs and test has 120 documents per rare class: read small gaps with care.")

GPU: Tesla T4 | CUDA 13.0 | cc 7.5

Linear attention, per seed
                          seed0    seed1    seed2
train_seconds           36.7300  36.0540  36.9512
final_val_acc            0.9793   0.9827   0.9827
overall_test_acc         0.9817   0.9830   0.9810
acc_general_dry          0.9912   0.9873   0.9814
acc_retail_pallet        0.9782   0.9846   0.9769
acc_refrigerated         0.9850   0.9817   0.9883
acc_fragile_high_value   0.9694   0.9833   0.9806
acc_hazmat               0.9500   0.9417   0.9583
acc_oversize_heavy       0.9750   0.9833   0.9917
rare_macro_recall        0.9625   0.9625   0.9750

Baseline vs linear attention: mean and sample std (ddof=1) over seeds 0, 1, 2
                   baseline_mean  baseline_std  linear_mean  linear_std  linear_minus_baseline
overall_test_acc          0.9763        0.0018       0.9819      0.0010                 0.0056
hazmat_recall             0.9500        0.0220       0.9500      0.0083                 0.0000
oversize_recall        

## Verification checks

In [15]:
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))

bcfg, lcfg = base["config"], lin["config"]

# 1. same dataset
check("Dataset SHA-256: baseline run == linear run == reference",
      bcfg["data_sha256"] == lcfg["data_sha256"] == DATA_SHA256 == REFERENCE_DATA_SHA256, DATA_SHA256[:16] + "...")
check("Vocab size, class counts and token counts match the baseline run",
      bcfg["vocab"] == VOCAB and base["class_counts"] == dist
      and base["token_stats"]["test"]["nonpad_tokens"] == TOKSTATS["test"]["nonpad_tokens"],
      f"vocab {VOCAB}, test nonpad tokens {TOKSTATS['test']['nonpad_tokens']}")

# 2. same test examples
check("Same test examples (same data hash; identical test class supports in every run)",
      all(B[s]["rare_support"] == L[s]["rare_support"] for s in L) and base["class_counts"]["test"] == dist["test"]
      and len(Y["test"]) == sum(dist["test"]), f"{len(Y['test'])} test docs, rare support {L['seed0']['rare_support']}")

# 3. sequence length
check("Both used sequence length 128", bcfg["max_len"] == lcfg["max_len"] == MAX_LEN == 128
      and all(X[k].shape[1] == 128 for k in X), str({k: tuple(X[k].shape) for k in X}))

# 4. identical training settings
keys = ["epochs", "batch", "lr", "wd", "warmup", "max_len", "d_model", "layers", "heads", "d_ff", "vocab",
        "train_label_noise", "data_seed"]
check("Identical training and architecture settings (all config fields)",
      all(bcfg[k] == lcfg[k] for k in keys) and set(B) == set(L) == {f"seed{s}" for s in SEEDS},
      ", ".join(f"{k}={lcfg[k]}" for k in keys[:6]))
check("Same initial weights as the baseline for each seed", INIT_IDENTICAL)

# 5. padding excluded
base_pad = None
try:
    _bm = build_baseline().to(DEVICE)
    _bm.load_state_dict(torch.load(f"{OUT_DIR}/baseline_seed0.pt", map_location=DEVICE))
    base_pad = padding_invariance(_bm, X["val"][:64]); del _bm
except FileNotFoundError:
    pass
lin_pad = max(r["padding_test_trained_max_logit_diff"] for r in L.values())
check("Padding excluded: linear (trained) logit change < 1e-4 when trailing padding changes",
      lin_pad < 1e-4 and PAD_TEST_UNTRAINED < 1e-4, f"trained max {lin_pad:.2e}, untrained {PAD_TEST_UNTRAINED:.2e}")
check("Fast linear attention matches the explicit masked T x T reference", REF_TEST < 1e-4, f"{REF_TEST:.2e}")
check("Padding test detects leakage (negative control with masking off fails the test)", LEAK_CONTROL > 1e-4,
      f"control diff {LEAK_CONTROL:.2e}")
check("Padding excluded: baseline (trained seed 0) logit change < 1e-4",
      base_pad is not None and base_pad < 1e-4, "checkpoint not found" if base_pad is None else f"{base_pad:.2e}")

# 6. val/test labels are clean: regenerate with zero label noise and compare
_noise = TRAIN_LABEL_NOISE
try:
    TRAIN_LABEL_NOISE = 0.0
    _clean, _, _ = build_dataset()
finally:
    TRAIN_LABEL_NOISE = _noise
check("Validation and test labels and texts untouched by label noise",
      all(_clean[k][0] == data[k][0] and np.array_equal(_clean[k][1], data[k][1]) for k in ("val", "test")))
check("Label noise only in train (3% target)",
      int((data["train"][1] != y_train_clean).sum()) == int(flipped.sum()) > 0,
      f"{int(flipped.sum())} of {len(flipped)} train labels flipped ({flipped.mean():.2%})")
del _clean

# 7. no test data in training or selection
overlap = len(set(data["train"][0]) & set(data["test"][0])) + len(set(data["val"][0]) & set(data["test"][0]))
check("No identical document text shared between test and train/val", overlap == 0, f"{overlap} shared")
check("Test evaluated exactly once per linear model, after training; final-epoch model used",
      TEST_EVAL_CALLS == len(SEEDS) and all(len(r["epoch_log"]) == EPOCHS for r in L.values()),
      f"{TEST_EVAL_CALLS} test evaluations; selection policy: last epoch, val only monitored")

# 8. parameters
check("Parameter counts within 2%", pct <= 2.0, f"{pb['total']:,} vs {pl['total']:,} ({pct:.4f}%)")

print(f"{'check':<92} result")
for name, ok, detail in checks:
    print(f"{name:<92} {'PASS' if ok else 'FAIL'}  {detail}")
assert all(ok for _, ok, _ in checks), "one or more checks failed"
print("\nAll checks passed.")

check                                                                                        result
Dataset SHA-256: baseline run == linear run == reference                                     PASS  e74c4242ce78d13f...
Vocab size, class counts and token counts match the baseline run                             PASS  vocab 5565, test nonpad tokens 131019
Same test examples (same data hash; identical test class supports in every run)              PASS  3000 test docs, rare support {'hazmat': 120, 'oversize_heavy': 120}
Both used sequence length 128                                                                PASS  {'train': (10500, 128), 'val': (1500, 128), 'test': (3000, 128)}
Identical training and architecture settings (all config fields)                             PASS  epochs=8, batch=64, lr=0.001, wd=0.01, warmup=100, max_len=128
Same initial weights as the baseline for each seed                                           PASS  
Padding excluded: linear (trained) logit change < 1